<div align="center">

# 🩺 Patient Segmentation using Unsupervised Machine Learning
### A Clustering-Based Analysis of the NHANES Health & Nutrition Survey

---

**Master's Level Machine Learning — Mini Project**

</div>

| | |
|---|---|
| **Project Title** | Patient Segmentation using Unsupervised Machine Learning on NHANES Data |
| **Student Name** | Syed Khaja Irfan Uddin |
| **University** | PES University |
| **Course** | Unsupervised Machine Learning / Machine Learning (Master's) |
| **Project Objective** | Discover clinically meaningful patient sub-populations ("segments") from demographic, dietary, and laboratory data using clustering algorithms, and translate them into actionable healthcare insights. |
| **Dataset** | National Health and Nutrition Examination Survey (NHANES), NCHS / CDC |
| **Software Used** | Python 3, Jupyter Notebook |
| **Primary Metric** | Silhouette Score |
| **Dimensionality Rule** | Any dimensionality reduction must retain **≥ 95%** of variance |

---

#### Python Libraries Used
`numpy` · `pandas` · `matplotlib` · `seaborn` · `plotly` · `scikit-learn` · `scipy` · `yellowbrick` · `kneed` *(and optionally `umap-learn`)*

<div align="center">
<i>"Not all patients are the same — and treating them as if they were is the central failure of population-scale medicine.<br>Clustering lets the data tell us who belongs together."</i>
</div>

# 1 · Introduction

## 1.1 What is Machine Learning?

**Machine Learning (ML)** is the branch of Artificial Intelligence in which algorithms *learn patterns directly from data* instead of being explicitly programmed with hand-written rules. Formally (Tom Mitchell, 1997):

> A computer program is said to learn from experience **E** with respect to some task **T** and performance measure **P**, if its performance at **T**, as measured by **P**, improves with experience **E**.

In this project:
- **Task (T)** → grouping patients into health segments,
- **Experience (E)** → the NHANES survey records,
- **Performance (P)** → the **Silhouette Score** (cohesion vs. separation of clusters).

## 1.2 Types of Machine Learning

| Paradigm | Labels? | Goal | Example |
|---|---|---|---|
| **Supervised** | Yes (X → y) | Predict a known target | Predict diabetes (yes/no) |
| **Unsupervised** | No (X only) | Discover hidden structure | **Segment patients (this project)** |
| **Semi-supervised** | Few labels | Leverage a little labelled data | Label a few scans, propagate |
| **Reinforcement** | Reward signal | Learn a policy by trial & error | Adaptive treatment dosing |

### Supervised Learning
The model sees input–output pairs $(x_i, y_i)$ and learns a mapping $f: X \rightarrow y$ that minimises a loss $\mathcal{L}(y, f(x))$. Requires a **ground-truth label**.

### Unsupervised Learning
The model sees **only inputs** $\{x_1, x_2, \dots, x_n\}$ with **no labels**, and must uncover the intrinsic geometry of the data — density, groupings, manifolds. **Clustering** is the flagship unsupervised task, and the heart of this project. Because NHANES has *no "patient type" column*, patient segmentation is inherently unsupervised.

## 1.3 Clustering

**Clustering** partitions $n$ observations into $k$ groups so that points within a group are *similar* and points in different groups are *dissimilar*. "Similarity" is quantified by a **distance metric** — most commonly the Euclidean distance:

$$ d(\mathbf{x}, \mathbf{y}) = \sqrt{\sum_{j=1}^{p} (x_j - y_j)^2} $$

A "good" clustering minimises **intra-cluster** distance and maximises **inter-cluster** distance.

### Why clustering is important
- Reveals structure **without labels** — ideal when outcomes are unknown or expensive to obtain.
- Compresses a complex population into a handful of interpretable **archetypes**.
- Serves as a foundation for downstream personalisation, anomaly detection, and resource planning.

## 1.4 Real-World Applications
Customer/market segmentation · document & image grouping · fraud & anomaly detection · gene-expression subtyping · recommendation systems · **and healthcare patient stratification**.

## 1.5 Healthcare Analytics & Patient Segmentation

- **Healthcare analytics** turns raw clinical, dietary, and lab data into decisions that improve outcomes and lower cost.
- **Patient segmentation** groups patients by shared health profiles (metabolic, cardiovascular, nutritional). Unlike one-size-fits-all care, segments enable *targeted* action.
- **Precision medicine** — "the right treatment, for the right patient, at the right time." Segments are the population-level scaffold on which precision medicine is built.
- **Population health management** — payers and public-health bodies allocate scarce resources (screening, clinics, budgets) to the segments that need them most.
- **Preventive healthcare** — identifying *rising-risk* segments (e.g. pre-diabetic, dyslipidemic) enables intervention *before* disease onset, where impact is highest and cost is lowest.

## 1.6 Expected Outcome of this Project
A set of **data-driven, clinically interpretable patient segments** (e.g. *Healthy Young Adults*, *Metabolic-Risk*, *High-Cardiovascular-Risk Seniors*), each with a quantitative profile and a set of recommended healthcare interventions — validated by the **Silhouette Score** and stress-tested across multiple clustering algorithms.

## 1.7 Environment Setup & Global Configuration

**What we are doing:** importing every library, fixing a global random seed for reproducibility, and configuring a consistent, publication-quality visual style.

**Why it is required:** A fixed `RANDOM_STATE` guarantees that stochastic steps (K-Means initialisation, sampling, t-SNE) produce identical results on every run — a non-negotiable requirement for academic reproducibility. Centralised styling ensures every figure shares fonts, sizes, and a colour palette.

In [ ]:
# ============================== CORE STACK ==============================
import os                     # file-system paths
import time                   # execution timing (used in the Appendix)
import warnings               # silence non-critical library warnings
import numpy as np            # numerical arrays / linear algebra
import pandas as pd           # tabular data handling

# ============================== VISUALISATION ===========================
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

# ============================== SCIKIT-LEARN ============================
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import VarianceThreshold
from sklearn.decomposition import PCA
from sklearn.cluster import (KMeans, DBSCAN, AgglomerativeClustering,
                             SpectralClustering)
from sklearn.mixture import GaussianMixture
from sklearn.metrics import (silhouette_score, silhouette_samples,
                             davies_bouldin_score, calinski_harabasz_score)
from sklearn.manifold import TSNE

# ============================== SCIPY / STATS ===========================
from scipy import stats
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import cdist

warnings.filterwarnings('ignore')          # keep the notebook output clean

# ---- Global reproducibility seed ----
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# ---- Publication-quality plotting defaults ----
sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams.update({
    'figure.figsize'  : (10, 5),   # default canvas size
    'figure.dpi'      : 110,       # crisp on screen
    'axes.titlesize'  : 13,
    'axes.titleweight': 'bold',
    'axes.labelsize'  : 11,
    'axes.grid'       : True,
    'grid.alpha'      : 0.3,
    'font.family'     : 'DejaVu Sans',
})
# A consistent qualitative palette reused for cluster colours throughout.
PALETTE = ['#2E86AB', '#E4572E', '#17A398', '#F4A261',
           '#8367C7', '#5C946E', '#D7263D', '#3D5A80']
sns.set_palette(PALETTE)
pio.templates.default = 'plotly_white'

# ---- Project directory ----

ROOT      = os.path.abspath('..')
DATA_DIR  = os.path.join(ROOT, 'data')
IMG_DIR   = os.path.join(ROOT, 'images')
MODEL_DIR = os.path.join(ROOT, 'models')
LOG_DIR   = os.path.join(ROOT, 'logs')
for _d in (DATA_DIR, IMG_DIR, MODEL_DIR, LOG_DIR):
    os.makedirs(_d, exist_ok=True)

# ---- Logging: write run logs to logs/ (and echo to the notebook) ----
import logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)s | %(message)s',
    handlers=[logging.FileHandler(os.path.join(LOG_DIR, 'run.log'), mode='w'),
              logging.StreamHandler()])
log = logging.getLogger('patient_segmentation')

# ---- Auto-save EVERY matplotlib figure to images/ (transparently) ----

_FIGN = {'n': 0}
_orig_show = plt.show
def _show_and_save(*args, **kwargs):
    _FIGN['n'] += 1
    try:
        fig = plt.gcf()
        fig.savefig(os.path.join(IMG_DIR, f'fig_{_FIGN["n"]:02d}.png'),
                    dpi=120, bbox_inches='tight')
    except Exception as _e:
        log.warning(f'figure save skipped: {_e}')
    return _orig_show(*args, **kwargs)
plt.show = _show_and_save

NB_START = time.time()          

log.info('Environment initialised | seed=%d', RANDOM_STATE)
print('Libraries imported successfully.')
print(f'pandas {pd.__version__} | numpy {np.__version__}')
print(f'Global RANDOM_STATE = {RANDOM_STATE}')
print(f'Project root : {ROOT}')
print(f'Figures -> images/ | models -> models/ | logs -> logs/run.log')

2026-07-21 12:01:15,292 | INFO | Environment initialised | seed=42


Libraries imported successfully.
pandas 2.3.3 | numpy 2.0.2
Global RANDOM_STATE = 42
Project root : /Users/syedirfanuddin/Documents/PES/Projects/UMLRS/Patient_segmentation
Figures -> images/ | models -> models/ | logs -> logs/run.log
